# Qual LLM Studio — Colab launcher

Run-all from the menu, then click the proxy link that appears in the **Launch** cell.

**Recommended runtime**: GPU (T4 is fine for 4B models). Edit > Notebook settings > Hardware accelerator > GPU.

Leave this tab open while you work — Colab will disconnect idle notebooks. The keepalive cell at the bottom helps but isn't bulletproof.

## 1. Install dependencies

In [ ]:
!pip install --quiet gradio rapidfuzz pypdf python-docx tqdm openai-gabriel nest_asyncio json_repair openpyxl pandas datasets pyarrow dspy-ai
print('deps installed')

## 2. Install and start Ollama

If Ollama is already installed (e.g. on a LUMI Jupyter node), set `OLLAMA_BIN` to its path and skip the install step. Otherwise the first run downloads ~150MB.

In [ ]:
import os, shutil, subprocess, time, requests

# Check if ollama is already on PATH
ollama_bin = shutil.which('ollama')
if ollama_bin is None:
    print('Installing Ollama...')
    subprocess.run(
        'curl -fsSL https://ollama.com/install.sh | sh',
        shell=True, check=True,
    )
    ollama_bin = shutil.which('ollama')
print(f'Using ollama at: {ollama_bin}')

# Start the server in the background if nothing's listening
def ollama_alive():
    try:
        return requests.get('http://localhost:11434', timeout=2).status_code in (200, 404)
    except Exception:
        return False

if not ollama_alive():
    print('Starting ollama server...')
    subprocess.Popen(
        ['ollama', 'serve'],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
    )
    for _ in range(30):
        if ollama_alive():
            break
        time.sleep(1)

print('Ollama server alive:', ollama_alive())

## 3. Pull a model

Recommended starting models (Q4_K_M, fits on a T4):
- `qwen3:4b` — fast, good at JSON, supports thinking modes
- `gemma3:4b` — strong multilingual baseline
- `llama3.2:3b` — smallest, fastest, lower quality

For multimodal recipes you'll need a vision/audio-capable model: `llama3.2-vision`, `qwen2.5-vl`, etc.

In [ ]:
MODEL = 'qwen3:4b'  # change as needed
import requests, time

# Pull (this may take a while on first run)
!ollama pull $MODEL

# Verify the model was actually installed.
# Ollama's CLI sometimes reports success when the pull failed silently;
# we check /api/tags to see what's really there.
time.sleep(1)
tags = requests.get('http://localhost:11434/api/tags', timeout=5).json()
names = [m['name'] for m in tags.get('models', [])]
if MODEL in names or MODEL.split(':')[0] + ':latest' in names:
    print(f'✓ {MODEL} ready')
else:
    print(f'✗ {MODEL} NOT installed. Got: {names}')
    print('Common causes: out of disk, GPU not detected, bad model name.')

## 4. Clone the app repo (or use a local copy)

In [ ]:
import os
REPO_PATH = '/content/qual-llm-studio'
if not os.path.exists(REPO_PATH):
    # Replace this with the real repo URL once published
    !git clone https://github.com/YOUR_ORG/qual-llm-studio $REPO_PATH
%cd $REPO_PATH
!ls

## 5. (Optional) mount Google Drive

Skip this cell if you don't need to save runs to Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 6. Launch the UI

After this cell runs, click the proxied URL that appears. The UI runs on port 7860 inside the Colab VM.

In [ ]:
from google.colab import output
from app.main import build_ui
import gradio as gr

ui = build_ui()
ui.queue().launch(
    server_name='0.0.0.0',
    server_port=7860,
    share=False,
    inline=False,
    prevent_thread_lock=True,
    theme=gr.themes.Soft(),
)

output.serve_kernel_port_as_window(7860, anchor_text='Open Qual LLM Studio')

## 7. (Optional) Keepalive

Run this cell to fight Colab's idle disconnect. It clicks an invisible button every 60s. Stop the cell when you're done.

In [ ]:
from IPython.display import display, Javascript
display(Javascript('''
function keepAlive() {
  const btn = document.querySelector('colab-toolbar-button#connect');
  if (btn) { btn.click(); }
}
setInterval(keepAlive, 60000);
console.log('keepalive armed');
'''))
print('Keepalive armed. Leave this tab open.')